# Recommender pipeline: run everything in order

This notebook **runs** the recommender from start to finish. It does not contain new logic. Each step calls something that
already exists in the project: a dbt model, a script in `recommender/`, or a Snowflake check. Every step has a comment saying what it is for.

**Order of the pipeline**

| # | Step | What it is for | Where it runs |
|---|---|---|---|
| 1 | Check environment | Packages installed, login found | here |
| 2 | dbt models | Build the split, training rows, buyer and listing features in Snowflake | `dbt_project/` |
| 3 | Tests | Make sure the metric and feature code works on small examples | main folder |
| 4 | Baselines | Score popularity and category popularity on validation buyers | main folder |
| 5 | ALS tuning | Choose ALS settings on validation buyers | main folder |
| 6 | LightGBM B tuning | Choose option B settings on validation buyers | main folder |
| 7 | LightGBM A, first tuning | History only: option A before it was re-tuned | main folder |
| 8 | Slate check | Test option A inside FINN's own lists, choose its settings | main folder |
| 9 | Final test | Train with the chosen settings, score all models on test buyers, once | main folder |
| 10 | Write scores | Put each model's top 10 lists into Snowflake | main folder |
| 11 | dbt beyond accuracy | Build the variety and popularity table from those lists | `dbt_project/` |
| 12 | Check in Snowflake | Confirm the tables hold what they should | here |
| 13 | Results | Print the final comparison | here |

**How to use it:** the switches in the next cell decide which steps run. **Run All** runs the ones set to `True`.

## Switches

Set a step to `False` to skip it. Steps 5 to 8 (tuning) are off by default, because their results already exist and
two of them need care before a re-run (see the warnings on those steps).

In [1]:
RUN = {
    "check_environment": True,
    "dbt_models": True,
    "tests": True,
    "baselines": True,
    "als_tuning": False,
    "lgbm_b_tuning": False,
    "lgbm_a_first_tuning": False,
    "slate_check": False,
    "final_test": True,
    "write_scores": True,
    "dbt_beyond_accuracy": True,
    "check_snowflake": True,
    "show_results": True,
}

## Setup

In [2]:
# Find the project folder, load the .env login, and define one helper (run) that
# executes a command, prints its output live, and stops the notebook if the command fails.

import os
import warnings

warnings.filterwarnings("ignore")              # hide warnings in this notebook
os.environ["PYTHONWARNINGS"] = "ignore"        # hide them in the scripts the notebook starts

import importlib
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd()
while not (ROOT / "dbt_project").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
load_dotenv(ROOT / ".env", override=True)      # dbt and the scripts read the login from these variables

RESULTS = ROOT / "analysis" / "results"
DBT_PROJECT = ROOT / "dbt_project"
DBT = str(Path(sys.executable).parent / "dbt")  # dbt installed next to this Python
if not Path(DBT).exists():
    DBT = "dbt"


def run(command, cwd=ROOT):
    """Run a command, print its output as it comes, raise an error if it fails."""
    print("$", " ".join(str(part) for part in command), f"   (in {cwd.name})")
    start = time.time()
    process = subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()
    print(f"\n-> finished in {(time.time() - start) / 60:.1f} min, exit code {process.returncode}")
    if process.returncode != 0:
        raise RuntimeError(f"Step failed (exit code {process.returncode}). See the output above.")


def python_module(name):
    """Command for: python -m <name>, run from the main folder."""
    return [sys.executable, "-m", name]


print("Project folder:", ROOT)

Project folder: /Users/mansi/Documents/Projects/finn-marketplace-analytics


## Step 1. Check environment

Purpose: catch a missing package or login now, instead of an hour into a run.

In [3]:
# Confirm the packages are installed and the Snowflake login variables are set.

if RUN["check_environment"]:
    print("Python:", sys.version.split()[0])
    for name in ["pandas", "numpy", "scipy", "implicit", "lightgbm", "sklearn",
                 "threadpoolctl", "snowflake.connector", "pyarrow", "dotenv", "pytest"]:
        try:
            importlib.import_module(name)
            print("ok      ", name)
        except ImportError:
            print("MISSING ", name)
    for name in ["SNOWFLAKE_ACCOUNT", "SNOWFLAKE_USER"]:
        print(name, "set" if os.environ.get(name) else "MISSING")
    has_login = os.environ.get("SNOWFLAKE_PASSWORD") or os.environ.get("SNOWFLAKE_PRIVATE_KEY_PATH")
    print("Password or key path:", "set" if has_login else "MISSING")
    print("FINN_DATABASE (where dbt tables are read):", os.environ.get("FINN_DATABASE", "DEV (default)"))

Python: 3.9.6
ok       pandas
ok       numpy
ok       scipy
ok       implicit
ok       lightgbm
ok       sklearn
ok       threadpoolctl
ok       snowflake.connector
ok       pyarrow
ok       dotenv
ok       pytest
SNOWFLAKE_ACCOUNT set
SNOWFLAKE_USER set
Password or key path: set
FINN_DATABASE (where dbt tables are read): DEV (default)


## Step 2. dbt models

Purpose: build the tables the Python code reads. Runs in `dbt_project/`.

- `int_rec_split`: gives every buyer a group (train, validation, test) and marks each slate as first 5 or later.
- `mart_rec_train`: one row per listing shown, with clicked or not.
- `mart_rec_buyer_features`: top category, top county, search share per buyer.
- `mart_rec_item_features`: times shown and times clicked per listing.

The Python code caches what it reads from Snowflake, so the cache is deleted after a rebuild.

In [4]:
# (Re)build the four dbt models the recommender reads, then clear the cached copies.

if RUN["dbt_models"]:
    run([DBT, "build", "--profiles-dir", ".", "--select",
         "int_rec_split", "mart_rec_train", "mart_rec_buyer_features", "mart_rec_item_features"],
        cwd=DBT_PROJECT)
    shutil.rmtree(ROOT / "recommender" / "cache", ignore_errors=True)
    print("Cleared recommender/cache")

$ /Users/mansi/Documents/Projects/finn-marketplace-analytics/.venv/bin/dbt build --profiles-dir . --select int_rec_split mart_rec_train mart_rec_buyer_features mart_rec_item_features    (in dbt_project)
22:06:34  Running with dbt=1.10.23
22:06:34  This version of dbt is deprecated and no longer receives regular patches, including for known bugs. We recommend upgrading to a newer supported version of dbt: https://docs.getdbt.com/docs/dbt-versions
22:06:34  Registered adapter: snowflake=1.10.2
22:06:35  Found 16 models, 68 data tests, 10 sources, 492 macros
22:06:35  
22:06:35  Concurrency: 4 threads (target='dev')
22:06:35  
22:06:45  1 of 4 START sql table model analytics_intermediate.int_rec_split .............. [RUN]
22:06:48  1 of 4 OK created sql table model analytics_intermediate.int_rec_split ......... [SUCCESS 1 in 3.53s]
22:06:48  2 of 4 START sql table model analytics_marts.mart_rec_buyer_features ........... [RUN]
22:06:48  3 of 4 START sql table model analytics_marts.mart_re

## Step 3. Tests

Purpose: check the metric, feature and ranking code on small hand-made examples, so a silent mistake shows up as a failed test.

In [5]:
# Run the unit tests (recommender/tests/) before any long run.

if RUN["tests"]:
    run(python_module("pytest") + ["recommender", "-q"])

$ /Users/mansi/Documents/Projects/finn-marketplace-analytics/.venv/bin/python -m pytest recommender -q    (in finn-marketplace-analytics)
.............                                                            [100%]
13 passed in 0.74s

-> finished in 0.0 min, exit code 0


## Step 4. Baselines

Purpose: measure the simple recommenders we must beat (popularity, category popularity) on **validation** buyers.
Saves `rec_baselines_validation.csv`.

In [6]:
# Score popularity and category popularity on validation buyers (hit, NDCG, recall at 10).

if RUN["baselines"]:
    run(python_module("recommender.run_baselines"))

$ /Users/mansi/Documents/Projects/finn-marketplace-analytics/.venv/bin/python -m recommender.run_baselines    (in finn-marketplace-analytics)
              model    metric  value  ci_low  ci_high  buyers
         popularity    hit@10 0.0028  0.0018   0.0038   10964
         popularity   ndcg@10 0.0015  0.0009   0.0020   10964
         popularity recall@10 0.0016  0.0012   0.0019   10964
category_popularity    hit@10 0.0032  0.0021   0.0042   10964
category_popularity   ndcg@10 0.0018  0.0012   0.0025   10964
category_popularity recall@10 0.0023  0.0020   0.0027   10964

-> finished in 0.3 min, exit code 0


## Step 5. ALS tuning  (off by default)

Purpose: choose ALS settings (factors, regularisation, alpha) on **validation** buyers, by NDCG@10.
Saves `rec_als_tuning_validation.csv` and `rec_als_best_params.json`.

**Warning before switching this on:** the lists `FACTORS`, `REGULARIZATIONS` and `ALPHAS` at the top of `recommender/run_als.py`
hold only the last grid that was run. The best settings found (256 factors, alpha 300) came from earlier, wider grids.
Re-running with the lists as they are would overwrite the best settings with a worse pick.
Put the values you want to compare into those lists first, for example `ALPHAS = [100.0, 300.0, 500.0]`.

In [7]:
# Grid search for ALS settings on validation buyers; saves the best settings to a JSON file.

if RUN["als_tuning"]:
    run(python_module("recommender.run_als"))

## Step 6. LightGBM option B tuning  (off by default)

Purpose: option B trains on ALS's own 200 candidates for train buyers. This tries several LightGBM settings on
**validation** buyers and saves the best to `rec_lgbm_b_best_params.json`. Needs step 5's ALS settings.

In [8]:
# Choose the number of leaves and trees for option B on validation buyers.

if RUN["lgbm_b_tuning"]:
    run(python_module("recommender.run_lgbm_b"))

## Step 7. LightGBM option A, first tuning  (history only, off by default)

Purpose: option A trains on the listings FINN showed. This is its first tuning, scored on the ALS shortlist.
It was superseded by step 8 and is kept only to reproduce the record. Saves `rec_lgbm_a_early_params.json`.

In [9]:
# Option A scored on the ALS shortlist (the test that showed it was worse than ALS alone).

if RUN["lgbm_a_first_tuning"]:
    run(python_module("recommender.run_lgbm_a"))

## Step 8. Slate check for option A  (off by default)

Purpose: the fair test for option A. Inside the lists FINN really showed, does it put the clicked listing higher than
FINN's own order (within-slate NDCG)? Tunes A on that test and saves `rec_lgbm_a_best_params.json`, which the final test uses.

In [10]:
# Compare option A with FINN's order, random, popularity and ALS score inside real slates (validation buyers).

if RUN["slate_check"]:
    run(python_module("recommender.run_slate_check"))

## Step 9. Final test

Purpose: the one official evaluation. Trains ALS, option B and option A with the saved settings, builds every model's
top 10 for the **test** buyers, scores them, compares them, and saves the lists (with the fallback flag).
Takes about 25 minutes. Saves four `rec_eval_test_*.csv` files.

The next cell first checks that `GROUP = "test"` in `recommender/run_final.py`. If it says `"validation"`, the run would
write validation files and the later steps would not find the test lists.

In [11]:
# Final scoring on test buyers, run once with frozen settings. 
# Compares five recommenders: two simple rules (popularity, category popularity) and 
# three built on the two trained models (ALS alone, ALS + LightGBM option A, ALS + LightGBM option B).

if RUN["final_test"]:
    source = (ROOT / "recommender" / "run_final.py").read_text()
    assert 'GROUP = "test"' in source, 'Set GROUP = "test" on line 29 of recommender/run_final.py, then run again.'
    for name in ["rec_als_best_params.json", "rec_lgbm_b_best_params.json", "rec_lgbm_a_best_params.json"]:
        assert (RESULTS / name).exists(), f"Missing {name}. Run the tuning steps first."
    run(python_module("recommender.run_final"))

$ /Users/mansi/Documents/Projects/finn-marketplace-analytics/.venv/bin/python -m recommender.run_final    (in finn-marketplace-analytics)
GROUP: test
trained option B on 6088200 rows
trained option A on 1751279 rows
Buyers without an ALS list (popularity fallback): 0.0131

Top-10 lists, test buyers (22349 buyers):
              model    metric  value  ci_low  ci_high  buyers
         popularity    hit@10 0.0038  0.0030   0.0046   22349
         popularity   ndcg@10 0.0018  0.0014   0.0023   22349
         popularity recall@10 0.0018  0.0016   0.0021   22349
category_popularity    hit@10 0.0047  0.0038   0.0056   22349
category_popularity   ndcg@10 0.0023  0.0018   0.0028   22349
category_popularity recall@10 0.0025  0.0022   0.0028   22349
                als    hit@10 0.0476  0.0448   0.0504   22349
                als   ndcg@10 0.0255  0.0239   0.0272   22349
                als recall@10 0.0244  0.0234   0.0255   22349
         als_lgbm_a    hit@10 0.0159  0.0142   0.0175   22349
  

## Step 10. Write scores to Snowflake

Purpose: store each model's top 10 for every test buyer in `RAW.RECOMMENDER.SCORES`
(columns: split_group, model, user_id, item_id, rank, score, is_fallback). **Replaces the table** each time.

In [12]:
# Load the top 10 lists from step 9 into Snowflake so dbt and dashboards can use them.

if RUN["write_scores"]:
    run(python_module("recommender.write_scores"))

$ /Users/mansi/Documents/Projects/finn-marketplace-analytics/.venv/bin/python -m recommender.write_scores    (in finn-marketplace-analytics)
Rows in file: 1117450
Loaded: True rows: 1117450

Rows per model:
  als                    rows   223490   buyers   22349   fallback    293
  als_lgbm_a             rows   223490   buyers   22349   fallback    293
  als_lgbm_b             rows   223490   buyers   22349   fallback    293
  category_popularity    rows   223490   buyers   22349   fallback    293
  popularity             rows   223490   buyers   22349   fallback      0

-> finished in 0.2 min, exit code 0


## Step 11. dbt beyond accuracy

Purpose: build `mart_rec_beyond_accuracy`, one row per model: buyers scored, fallback share, catalogue coverage
(how many different listings it recommends) and the share of recommendations on the 1% most-clicked listings.
Runs in `dbt_project/`.

In [13]:
# Turn the stored lists into variety and popularity numbers per model.

if RUN["dbt_beyond_accuracy"]:
    run([DBT, "build", "--profiles-dir", ".", "--select", "mart_rec_beyond_accuracy"], cwd=DBT_PROJECT)

$ /Users/mansi/Documents/Projects/finn-marketplace-analytics/.venv/bin/dbt build --profiles-dir . --select mart_rec_beyond_accuracy    (in dbt_project)
22:21:59  Running with dbt=1.10.23
22:21:59  This version of dbt is deprecated and no longer receives regular patches, including for known bugs. We recommend upgrading to a newer supported version of dbt: https://docs.getdbt.com/docs/dbt-versions
22:22:00  Registered adapter: snowflake=1.10.2
22:22:00  Found 16 models, 68 data tests, 10 sources, 492 macros
22:22:00  
22:22:00  Concurrency: 4 threads (target='dev')
22:22:00  
22:22:01  1 of 4 START sql table model analytics_marts.mart_rec_beyond_accuracy .......... [RUN]
22:22:03  1 of 4 OK created sql table model analytics_marts.mart_rec_beyond_accuracy ..... [SUCCESS 1 in 2.20s]
22:22:03  2 of 4 START test not_null_mart_rec_beyond_accuracy_buyers_scored .............. [RUN]
22:22:03  3 of 4 START test not_null_mart_rec_beyond_accuracy_catalogue_coverage ......... [RUN]
22:22:03  4 of 4

## Step 12. Check in Snowflake

Purpose: confirm the tables hold what they should. Expect 5 models with the same number of buyers and 10 rows per buyer;
popularity has 0 fallback buyers.

In [14]:
# Read back the scores table and the beyond-accuracy table from Snowflake.

if RUN["check_snowflake"]:
    from analysis.snowflake_io import query

    scores = query('''
        select model, count(*) as rows_, count(distinct user_id) as buyers,
               count(distinct iff(is_fallback, user_id, null)) as fallback_buyers
        from RAW.RECOMMENDER.SCORES
        group by model
        order by model
    ''')
    display(scores)

    beyond = query("select * from analytics_marts.mart_rec_beyond_accuracy order by model")
    display(beyond)

,model,rows_,buyers,fallback_buyers
0,als,223490,22349,293
1,als_lgbm_a,223490,22349,293
2,als_lgbm_b,223490,22349,293
3,category_popularity,223490,22349,293
4,popularity,223490,22349,0


,split_group,model,buyers_scored,fallback_share,catalogue_items,distinct_items,catalogue_coverage,top1pct_popular_share
0,test,als,22349,0.01311,190819,56909,0.298236,0.375838
1,test,als_lgbm_a,22349,0.01311,190819,37116,0.194509,0.563251
2,test,als_lgbm_b,22349,0.01311,190819,65392,0.342691,0.310922
3,test,category_popularity,22349,0.01311,190819,64,0.000335,1.000000
4,test,popularity,22349,0.00000,190819,12,0.000063,1.000000


## Step 13. Results

Purpose: print the final comparison from the files step 9 saved: hit, NDCG and recall at 10 for each model, and the paired
differences (first model minus second, with a 95% range; if the range excludes zero, the difference is real).

In [15]:
# Show the final numbers without opening the CSV files.

if RUN["show_results"]:
    top10 = pd.read_csv(RESULTS / "rec_eval_test_top10.csv")
    display(top10.pivot(index="model", columns="metric", values="value").round(4))
    display(pd.read_csv(RESULTS / "rec_eval_test_differences.csv").round(4))
    display(pd.read_csv(RESULTS / "rec_eval_test_slates.csv").round(4))

metric,hit@10,ndcg@10,recall@10
model,,,
als,0.0476,0.0255,0.0244
als_lgbm_a,0.0159,0.0071,0.0091
als_lgbm_b,0.0536,0.0284,0.0265
category_popularity,0.0047,0.0023,0.0025
popularity,0.0038,0.0018,0.0018


,comparison,metric,difference,ci_low,ci_high
0,als minus category_popularity,hit@10,0.0429,0.0400,0.0457
1,als minus category_popularity,ndcg@10,0.0232,0.0215,0.0249
2,als minus category_popularity,recall@10,0.0219,0.0208,0.0230
3,als_lgbm_b minus als,hit@10,0.0060,0.0041,0.0078
4,als_lgbm_b minus als,ndcg@10,0.0029,0.0019,0.0039
5,als_lgbm_b minus als,recall@10,0.0021,0.0015,0.0027
6,als_lgbm_a minus als,hit@10,-0.0317,-0.0345,-0.0289
7,als_lgbm_a minus als,ndcg@10,-0.0184,-0.0200,-0.0168
8,als_lgbm_a minus als,recall@10,-0.0153,-0.0163,-0.0144
9,als_lgbm_b minus als_lgbm_a,hit@10,0.0377,0.0348,0.0406


,model,ndcg,ci_low,ci_high,diff_vs_finn,diff_low,diff_high,buyers
0,finn_order,0.5160,0.5125,0.5195,0.0000,0.0000,0.0000,9701
1,random_order,0.5001,0.4967,0.5036,-0.0159,-0.0200,-0.0117,9701
2,popularity,0.5286,0.5249,0.5323,0.0126,0.0089,0.0164,9701
3,als_score_only,0.5205,0.5169,0.5241,0.0045,0.0007,0.0083,9701
4,lgbm_a,0.5172,0.5136,0.5208,0.0012,-0.0026,0.0050,9701
